## 1. Important Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd 
import numpy as np 
import seaborn as se 
import matplotlib as plt
import random 
import plotly.express as px

## 2. Createting PySpark Session

In [0]:
spark = SparkSession.builder.appName('Data_Analysis_with_PySpark').getOrCreate()

## 3. Generating Data

In [0]:
names = [
    "Alice", "Bob", "Charlie", "David", "Eve", "Fiona", "George", "Hannah",
    "Ivy", "Jack", "Kaitlyn", "Liam", "Olivia", "Liam", "Emma", "Noah", 
    "Ava", "Oliver", "Charlotte", "Elijah", "Sophia", "James", "Amelia", 
    "Benjamin", "Isabella", "Lucas", "Mia", "Mason", "Harper", "Ethan", 
    "Evelyn", "Alexander", "Abigail", "Henry", "Ella", "Jackson", "Scarlett", 
    "Aiden", "Grace", "Samuel", "Lily", "Sebastian"
]
genders = ["Male", "Female", None]
subjects = ["Math", "Science", "History", "English", "Art", "PE", None]
cities = [
    "New York", "Los Angeles", "Chicago", "Houston", 
    "Bangalore", "Hajipur", "Sitamardhi", "MP", None
]
states = ["NY", "CA", "IL", "TX", "Bihar", "Karnataka", "Sitamardhi", None]
countries = ["USA", "India", "Pakistan", "Nepal", "China", None]
graduated_status = ["Yes", "No", None]

data = [
    (
        i, 
        random.choice(names),  # student_name
        random.choice([random.randint(18, 25), None]),  # age
        random.choice(genders),  # gender
        random.choice(subjects),  # subject
        random.choice([random.randint(50, 100), None]),  # marks
        random.choice(cities),  # city
        random.choice(states),  # state
        random.choice(countries),  # country
        random.choice(graduated_status),  # graduated
    )
    for i in range(1, 501)
]

## 4. Creating a DATAFRAME

In [0]:
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("marks", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("graduated", StringType(), True),
])

df = spark.createDataFrame(data,schema=schema)
df.show(5)

+----------+------------+----+------+-------+-----+-----------+---------+--------+---------+
|student_id|student_name| age|gender|subject|marks|       city|    state| country|graduated|
+----------+------------+----+------+-------+-----+-----------+---------+--------+---------+
|         1|       David|null|  Male|History|   87|  Bangalore|    Bihar|   Nepal|      Yes|
|         2|       David|  22|  Male|History| null|    Hajipur|Karnataka|   China|       No|
|         3|      Olivia|  21|  null|    Art|   94|    Houston|       CA|   China|      Yes|
|         4|       Grace|  21|  null|Science|   93|    Chicago|       NY|   Nepal|       No|
|         5|   Alexander|null|  Male|     PE| null|Los Angeles|     null|Pakistan|     null|
+----------+------------+----+------+-------+-----+-----------+---------+--------+---------+
only showing top 5 rows



## 5.OverView of DataFrame

In [0]:
df.show(10)

+----------+------------+----+------+-------+-----+--------+---------+--------+---------+
|student_id|student_name| age|gender|subject|marks|    city|    state| country|graduated|
+----------+------------+----+------+-------+-----+--------+---------+--------+---------+
|         1|        Noah|null|  null|Science| null|    null|       IL|    null|      Yes|
|         2|        Liam|null|Female|   null| null| Chicago|       NY|Pakistan|      Yes|
|         3|    Isabella|  18|  Male|     PE|   76|    null|       CA|    null|     null|
|         4|        Liam|null|  null|History|   79|New York|     null|   India|      Yes|
|         5|   Charlotte|  22|Female|History| null| Hajipur|Karnataka|Pakistan|      Yes|
|         6|       Mason|  25|  Male|   Math|   50| Chicago|       NY|   India|      Yes|
|         7|       Mason|  20|  null|History| null| Chicago|       IL|     USA|     null|
|         8|       Fiona|  20|Female|   Math| null|New York|       NY|    null|       No|
|         

In [0]:
df.describe().show()

+-------+-----------------+------------+------------------+------+-------+------------------+----------+-----+-------+---------+
|summary|       student_id|student_name|               age|gender|subject|             marks|      city|state|country|graduated|
+-------+-----------------+------------+------------------+------+-------+------------------+----------+-----+-------+---------+
|  count|              500|         500|               246|   325|    428|               247|       448|  442|    425|      337|
|   mean|            250.5|        null|21.715447154471544|  null|   null| 74.55465587044534|      null| null|   null|     null|
| stddev|144.4818327679989|        null|2.2260803850870756|  null|   null|14.283570092109864|      null| null|   null|     null|
|    min|                1|     Abigail|                18|Female|    Art|                50| Bangalore|Bihar|  China|       No|
|    max|              500|      Sophia|                25|  Male|Science|               100|Sita

In [0]:
df.dtypes

Out[7]: [('student_id', 'int'),
 ('student_name', 'string'),
 ('age', 'int'),
 ('gender', 'string'),
 ('subject', 'string'),
 ('marks', 'int'),
 ('city', 'string'),
 ('state', 'string'),
 ('country', 'string'),
 ('graduated', 'string')]

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- graduated: string (nullable = true)



In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,175,72,0,52,58,75,163


## 6. Replacing Null Values

In [0]:
df = df.fillna({
    'gender' : 'UniSex',
    'subject' : 'Hindi',
    'city' : 'Kalitand',
    'State' : 'Others',
    'Country' : 'India',
    'graduated' : 'Failed'
})

In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,0,0,0,0,0,0,0


## 7. Checking duplicate Values in each column

In [0]:
for index,column in enumerate(df.columns):
    print(f"checking duplicates in the column: {column}")
    duplicate_count = df.groupBy(column).count().filter("count > 1 ")
    duplicate_count.show()


checking duplicates in the column: student_id
+----------+-----+
|student_id|count|
+----------+-----+
+----------+-----+

checking duplicates in the column: student_name
+------------+-----+
|student_name|count|
+------------+-----+
|       Lucas|   14|
|         Ivy|   19|
|    Isabella|   10|
|       James|    8|
|    Benjamin|   12|
|        Jack|   11|
|         Ava|   13|
|        Ella|   20|
|        Noah|   15|
|       Mason|   12|
|       Ethan|   15|
|     Charlie|    8|
|         Mia|   13|
|         Bob|   14|
|        Liam|   17|
|     Jackson|   19|
|   Sebastian|   12|
|      Elijah|    7|
|   Alexander|    7|
|       Alice|    9|
+------------+-----+
only showing top 20 rows

checking duplicates in the column: age
+----+-----+
| age|count|
+----+-----+
|  22|   28|
|null|  254|
|  20|   36|
|  19|   31|
|  23|   33|
|  25|   30|
|  24|   40|
|  21|   28|
|  18|   20|
+----+-----+

checking duplicates in the column: gender
+------+-----+
|gender|count|
+------+-----+
|Fe

In [0]:
duplicate_counts = []
# Loop through each column in the DataFrame
for column in df.columns:
    # Group by the column and count duplicates (count > 1)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,49
6,city,9
7,state,8
8,country,5
9,graduated,3


In [0]:
# Loop through each column in the DataFrame
for column in df.columns:
    print(column)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


student_id
student_name
age
gender
subject
marks
city
state
country
graduated


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,49
6,city,9
7,state,8
8,country,5
9,graduated,3


## 8.Descriptive Statistics and Basic Summarization

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- subject: string (nullable = false)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- country: string (nullable = false)
 |-- graduated: string (nullable = false)



In [0]:
# What are the central tendencies (mean, median, mode) of marks and age?
df.groupBy("subject").agg(
    mean("marks").alias("marks_mean"),
    mean("age").alias("mean_age")
).show()



+-------+-----------------+------------------+
|subject|       marks_mean|          mean_age|
+-------+-----------------+------------------+
|Science|          72.4375|21.071428571428573|
|    Art|           77.025|21.511111111111113|
|   Math|         75.34375| 21.72222222222222|
|English|72.19047619047619|21.923076923076923|
|History|         79.15625| 21.38235294117647|
|  Hindi|73.78787878787878|22.457142857142856|
|     PE|72.36111111111111|21.862068965517242|
+-------+-----------------+------------------+



In [0]:
# What is the spread of marks in each subject?
df.groupBy("subject").agg(
    min("marks").alias("min_marks"),
    max("marks").alias("max_marks"),
).show()

+-------+---------+---------+
|subject|min_marks|max_marks|
+-------+---------+---------+
|Science|       51|      100|
|    Art|       50|       97|
|   Math|       50|       96|
|English|       50|       97|
|History|       52|      100|
|  Hindi|       51|      100|
|     PE|       50|       93|
+-------+---------+---------+



In [0]:
# What is the standard deviation of marks for different cities or countries?
df.groupBy("city").agg(
    stddev("marks").alias("stddev_marks")
).show()

+-----------+------------------+
|       city|      stddev_marks|
+-----------+------------------+
|  Bangalore|14.840006955915248|
|Los Angeles|13.627849411036669|
| Sitamardhi| 15.58022250589937|
|    Chicago|15.256848579291486|
|    Hajipur|12.719458140369635|
|    Houston|13.547854252194687|
|   New York| 15.33282050424463|
|         MP|14.865428092757787|
|   Kalitand|13.668333231719709|
+-----------+------------------+



In [0]:
from pyspark.sql import functions as F

# Calculate the distribution of students in each subject
df.groupBy("subject").agg(
    F.count("student_id").alias("stu_count")  # Count the number of students in each subject
).show()


+-------+---------+
|subject|stu_count|
+-------+---------+
|Science|       61|
|    Art|       76|
|   Math|       74|
|English|       82|
|History|       69|
|  Hindi|       72|
|     PE|       66|
+-------+---------+



In [0]:
# How many students belong to each gender and how does their academic performance differ?
df.groupBy("gender").agg(
    F.count("*").alias("stu_count"),       # Count of students per gender
    F.avg("marks").alias("avg_marks"),     # Average marks per gender
    F.stddev("marks").alias("std_marks")   # Standard deviation of marks per gender
).show()


+------+---------+-----------------+------------------+
|gender|stu_count|        avg_marks|         std_marks|
+------+---------+-----------------+------------------+
|Female|      165|75.86206896551724| 14.82076828014638|
|UniSex|      175|74.36363636363636|13.640988051601077|
|  Male|      160|73.36144578313252|14.350642301638826|
+------+---------+-----------------+------------------+



In [0]:
# What is the distribution of students across different cities, states, and countries?
df.groupBy("city","state","country").count().show()

+-----------+----------+--------+-----+
|       city|     state| country|count|
+-----------+----------+--------+-----+
|    Chicago|Sitamardhi|   Nepal|    3|
|Los Angeles|        IL|   India|    2|
|   New York|        TX|   China|    3|
| Sitamardhi| Karnataka|     USA|    2|
|    Chicago|     Bihar|     USA|    1|
|   New York|        NY|   India|    4|
|         MP|Sitamardhi|   India|    3|
|   New York|     Bihar|Pakistan|    1|
|   New York|Sitamardhi|     USA|    1|
|    Chicago|        IL|   Nepal|    2|
|         MP|        IL|   India|    5|
| Sitamardhi|        TX|   China|    3|
|    Chicago|        IL|     USA|    3|
|   Kalitand|    Others|   India|    3|
|         MP|    Others|   China|    2|
|    Hajipur|        TX|     USA|    4|
|         MP|        NY|     USA|    1|
|   Kalitand|        CA|     USA|    1|
|   New York|        IL|     USA|    1|
|    Houston|     Bihar|   China|    2|
+-----------+----------+--------+-----+
only showing top 20 rows



In [0]:
# How many students are marked as graduated versus those who are not? What is the graduation rate?
graduated_status = df.groupBy("graduated").agg(F.count("*").alias("stu_count"))
total_stu = df.count()

graduated_status.withColumn("graducation_perc",col("stu_count")/total_stu * 100).show()


+---------+---------+------------------+
|graduated|stu_count|  graducation_perc|
+---------+---------+------------------+
|   Failed|      163|              32.6|
|       No|      161|              32.2|
|      Yes|      176|35.199999999999996|
+---------+---------+------------------+



In [0]:
graduated_status.show()

+---------+---------+
|graduated|stu_count|
+---------+---------+
|   Failed|      163|
|       No|      161|
|      Yes|      176|
+---------+---------+



##  9.Exploratory Data Analysis (EDA) : Marks Distribution

In [0]:
# What is the distribution of marks within each subject? Are there specific subjects with higher or lower marks?

marks_dis = df.groupBy("subject").agg(F.sum("marks").alias("total_marks"))
marks_dis = marks_dis.toPandas()

fig_bar = px.bar(
    marks_dis, x= 'subject',y = 'total_marks', # define x and y asis 
    title = "Total Marks Distribution By Subject", # Add Title into the Graph
    labels={"total_marks": "Total Marks", "subject": "Subject"},# display labels
    text='total_marks' , # Display total marks on the bars
        )
fig_bar.show()

fig_pie = px.pie(
    marks_dis,
    names = 'subject',
    values = 'total_marks',
    title = "Total Marks Distribution By Subject",
    labels = {"subject": "Subject", "total_marks": "Total Marks"},
    color_discrete_sequence = px.colors.qualitative.Pastel 
)
fig_pie.show()

In [0]:
# How do marks vary across different age groups or genders?
marks_by_age = df.filter(col("age").isNotNull()).groupBy("age").agg(sum("marks").alias("total_marks"))
marks_by_age = marks_by_age.toPandas()
#print(marks_by_age)

fig_bar = px.bar(
    marks_by_age,
    x = 'age',
    y = 'total_marks',
    title = 'Marks Distribution By Age',
    labels = {'total_marks' : 'Total Marks' , 'subject':'Subjet'},
    text = 'total_marks'
)
fig_bar.show()

fig_pie = px.pie(
    marks_by_age,
    names = 'age',
    values = 'total_marks',
    title = "Total Marks Distribution By Subject",
    labels = {"age": "Age", "total_marks": "Total Marks"},
    color_discrete_sequence = px.colors.qualitative.Pastel 
)
fig_pie.show()

marks_by_gender = df.filter(col("gender").isNotNull()).groupBy("gender").agg(sum("marks").alias("total_marks"))
marks_by_gender = marks_by_gender.toPandas()

fig_bar_by_gender = px.bar(
    marks_by_gender,
    x =  'gender',
    y = 'total_marks',
    title = 'Marks Distribution By Gender',
    labels = {"gender" : "Gender","total_marks" : "Total_Marks"},
    text = 'total_marks'
)
fig_bar_by_gender.show()

fig_pie_by_gender = px.pie(
    marks_by_gender,
    names = 'gender',
    values = 'total_marks',
    title = 'Marks Distribution By Gender',
    color_discrete_sequence = px.colors.qualitative.Antique
)

fig_pie_by_gender.show()



## 10 .Exploratory Data Analysis (EDA) : Creating Stacked Bar Graph

In [0]:
# How does the age distribution look? Is it skewed or normal? Are there specific age groups that perform better academically?

df = df.withColumn(
    "age_group",
    F.when(F.col("age") < 18, "Minor")
     .when((F.col("age") >= 18) & (F.col("age") <= 30), "Young Adult")
     .otherwise("Adult")
)
student_by_age_and_gender = df.groupBy("age_group", "gender").agg(F.count("*").alias("stu_count"))
student_by_age_and_gender = student_by_age_and_gender.toPandas()

fig_stacked_bar = px.bar(
    student_by_age_and_gender,
    x = 'age_group',
    y = 'stu_count',
    color = 'gender',
    title = 'Distribution By Age and Gender',
    text = 'stu_count',
    color_discrete_sequence = px.colors.qualitative.Pastel
)
fig_stacked_bar.show()



## WORKING WITH GRAPHS


## 11. Marks Distribution (Histogram)

In [0]:
df1 = df.toPandas() 
fig = px.histogram(
    df1,
    x = 'marks',
    nbins= 20,
    title= 'Marks Distribution'
)
fig.update_layout(xaxis_title = 'marks',yaxis_title = 'Frequency')
fig.show()


## 12.Average Marks by Subject (Bar Chart)

In [0]:
avg_marks_by_subject = df.groupBy("subject").agg(F.mean("marks").alias("Avg_marks"))
#avg_marks_by_subject.show()
avg_marks_by_subject.=


+-------+-----------------+
|subject|        Avg_marks|
+-------+-----------------+
|Science|73.58695652173913|
|    Art|           73.875|
|   null|75.39473684210526|
|   Math|             78.3|
|English| 74.6470588235294|
|History|71.29268292682927|
|     PE|74.31428571428572|
+-------+-----------------+




## 13. Marks vs Age (Scatter Plot)


## 14. Gender Distribution (Pie Chart)


## 15.Average Marks by City (Horizontal Bar Chart)


## 16.Graduation Status by Subject (Stacked Bar Chart)


## 17.Correlation Heatmap